# Moosic — 06. Hybrid Pipeline (Agglomerative + Recursive K-Means)

**Idea:** Agglomerative (Ward, k=8) found 8 balanced, musically distinct superclusters — including two that independently replicated real minority genres DBSCAN also found (classical/instrumental, electronic-instrumental). Rather than running the recursive K-Means split/merge on the whole 5000-song dataset at once (notebook `02`), run it **once per Agglomerative supercluster** instead — splits start from an already-coherent group rather than the whole blended mass.

This notebook runs **both** the hybrid pipeline and the original single-stage pipeline side by side, on the same data and feature set, for a guaranteed apples-to-apples comparison rather than trusting numbers from an earlier session.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import pairwise_distances_argmin
from sklearn import set_config
import os

set_config(transform_output="pandas")
os.makedirs("../outputs", exist_ok=True)
RANDOM_STATE = 42

## 2. Load Data & Scale

In [ ]:
df = pd.read_csv("../data/5000_songs.csv")
df.columns = df.columns.str.strip()

features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']

scaler = MinMaxScaler().set_output(transform="pandas")
scaled_all = scaler.fit_transform(df[features])

TARGET_MIN, TARGET_MAX = 20, 60
MIN_SONGS_TO_SPLIT = 10
MAX_K_PER_SPLIT = 5

print(f"Shape: {df.shape}")

## 3. Reusable Recursive Split / Reassign / Merge Function

Same fixed logic as notebook `02` (distance-based nearest-neighbor merge with a size cap, reassignment before merge, not after) — generalized to run on any subset of the data, not just the full dataset. Reassignment and merging are scoped *within* whatever `indices` are passed in, so a hybrid-mode call never lets one Agglomerative supercluster borrow members from another.


In [ ]:
def merge_undersized_leaves(leaves, leaf_centroids, min_size, max_size):
    leaves = [list(l) for l in leaves]
    leaf_centroids = leaf_centroids.copy()
    changed = True
    while changed:
        changed = False
        sizes = [len(l) for l in leaves]
        for i, size in enumerate(sizes):
            if size < min_size and len(leaves) > 1:
                candidates = [j for j in range(len(leaves)) if j != i]
                dists = np.linalg.norm(leaf_centroids[candidates] - leaf_centroids[i], axis=1)
                order = np.argsort(dists)
                sorted_candidates = [candidates[o] for o in order]
                nearest = next(
                    (j for j in sorted_candidates if len(leaves[j]) + size <= max_size),
                    sorted_candidates[0]
                )
                leaves[nearest].extend(leaves[i])
                del leaves[i]
                leaf_centroids = np.delete(leaf_centroids, i, axis=0)
                changed = True
                break
    return leaves, leaf_centroids


def run_recursive_pipeline(indices, scaled_all, target_min, target_max,
                            min_songs_to_split, max_k_per_split, random_state=42):
    """Runs split -> reassign -> merge on just the given indices. Returns {index: local_cluster_id}."""
    leaves = []

    def recursive_split(idx_list):
        n = len(idx_list)
        subset_scaled = scaled_all.loc[idx_list]
        if n <= target_max or n < min_songs_to_split:
            leaves.append(idx_list)
            return
        k = min(max_k_per_split, max(2, n // target_max))
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = kmeans.fit_predict(subset_scaled)
        for c in range(k):
            child = [idx for idx, lab in zip(idx_list, labels) if lab == c]
            if child:
                recursive_split(child)

    recursive_split(list(indices))

    leaf_centroids = np.array([scaled_all.loc[leaf].mean(axis=0).values for leaf in leaves])

    # reassignment SCOPED to this subset only
    idx_order = list(scaled_all.loc[indices].index)
    final_labels = pairwise_distances_argmin(scaled_all.loc[idx_order].values, leaf_centroids)
    label_map = dict(zip(idx_order, final_labels))

    reassigned = [[idx for idx in idx_order if label_map[idx] == lbl] for lbl in sorted(set(final_labels))]
    reassigned_centroids = np.array([scaled_all.loc[m].mean(axis=0).values for m in reassigned])

    final_clusters, _ = merge_undersized_leaves(reassigned, reassigned_centroids, target_min, target_max)

    result = {}
    for local_label, members in enumerate(final_clusters):
        for idx in members:
            result[idx] = local_label
    return result

## 4. Baseline — Single-Stage Recursive Pipeline (Whole Dataset at Once)

In [ ]:
baseline_result = run_recursive_pipeline(
    list(df.index), scaled_all, TARGET_MIN, TARGET_MAX, MIN_SONGS_TO_SPLIT, MAX_K_PER_SPLIT, RANDOM_STATE
)
df['baseline_cluster'] = df.index.map(baseline_result)

baseline_sizes = df['baseline_cluster'].value_counts()
baseline_in_range = baseline_sizes[(baseline_sizes >= TARGET_MIN) & (baseline_sizes <= TARGET_MAX)]

print(f"Baseline: {len(baseline_sizes)} playlists")
print(f"In range: {len(baseline_in_range)} / {len(baseline_sizes)} "
      f"({100*len(baseline_in_range)/len(baseline_sizes):.1f}%)")

## 5. Stage 1 — Agglomerative Coarse Clustering (k=8, Ward)

In [ ]:
agg = AgglomerativeClustering(n_clusters=8, metric='euclidean', linkage='ward')
df['agg_supercluster'] = agg.fit_predict(scaled_all)

print(df['agg_supercluster'].value_counts().sort_index())

## 6. Stage 2 — Recursive Split Within Each Supercluster

In [ ]:
df['hybrid_cluster'] = None

for supercluster_id in sorted(df['agg_supercluster'].unique()):
    member_indices = df[df['agg_supercluster'] == supercluster_id].index.tolist()

    local_result = run_recursive_pipeline(
        member_indices, scaled_all, TARGET_MIN, TARGET_MAX,
        MIN_SONGS_TO_SPLIT, MAX_K_PER_SPLIT, RANDOM_STATE
    )

    # prefix with supercluster id so labels stay globally unique and traceable
    for idx, local_label in local_result.items():
        df.loc[idx, 'hybrid_cluster'] = f"{supercluster_id}-{local_label}"

    print(f"Supercluster {supercluster_id} ({len(member_indices)} songs) "
          f"-> {len(set(local_result.values()))} playlists")

hybrid_sizes = df['hybrid_cluster'].value_counts()
hybrid_in_range = hybrid_sizes[(hybrid_sizes >= TARGET_MIN) & (hybrid_sizes <= TARGET_MAX)]

print(f"\nHybrid total: {len(hybrid_sizes)} playlists")
print(f"In range: {len(hybrid_in_range)} / {len(hybrid_sizes)} "
      f"({100*len(hybrid_in_range)/len(hybrid_sizes):.1f}%)")

## 7. Direct Comparison

In [ ]:
comparison = pd.DataFrame({
    "Pipeline": ["Single-stage (K-Means only)", "Hybrid (Agglomerative + K-Means)"],
    "Playlists": [len(baseline_sizes), len(hybrid_sizes)],
    "In range %": [round(100*len(baseline_in_range)/len(baseline_sizes), 1),
                    round(100*len(hybrid_in_range)/len(hybrid_sizes), 1)],
    "Mean size": [round(baseline_sizes.mean(), 1), round(hybrid_sizes.mean(), 1)],
    "Min size": [baseline_sizes.min(), hybrid_sizes.min()],
    "Max size": [baseline_sizes.max(), hybrid_sizes.max()]
})
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
axes[0].hist(baseline_sizes, bins=25, color='steelblue', edgecolor='black')
axes[0].axvline(TARGET_MIN, color='red', linestyle='--')
axes[0].axvline(TARGET_MAX, color='red', linestyle='--')
axes[0].set_title(f'Single-stage ({len(baseline_sizes)} playlists)')
axes[0].set_xlabel('Playlist size'); axes[0].set_ylabel('Number of playlists')

axes[1].hist(hybrid_sizes, bins=25, color='seagreen', edgecolor='black')
axes[1].axvline(TARGET_MIN, color='red', linestyle='--')
axes[1].axvline(TARGET_MAX, color='red', linestyle='--')
axes[1].set_title(f'Hybrid ({len(hybrid_sizes)} playlists)')
axes[1].set_xlabel('Playlist size')

plt.tight_layout()
plt.savefig("../outputs/06_hybrid_vs_baseline_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Spot-Check: Does a Hybrid Playlist Trace Back to a Coherent Supercluster?

In [ ]:
# pick one playlist from inside the classical/instrumental supercluster (id 3, per notebook 05's
# interpretation) and confirm it stayed musically coherent after the finer split
classical_playlists = [c for c in hybrid_sizes.index if str(c).startswith("3-")]
sample_playlist = classical_playlists[0]

sample_songs = df[df['hybrid_cluster'] == sample_playlist]
print(f"Playlist {sample_playlist} ({len(sample_songs)} songs):")
print(sample_songs[features].mean().round(3))
print()
print(sample_songs['name'].sample(min(10, len(sample_songs)), random_state=1).to_string(index=False))

## 8a. The Real Test — Within-Playlist Coherence, Not Just Size Compliance

The size-target metric (Step 7) is near-ceiling for both pipelines, since both are specifically tuned to hit it — it can't show a meaningful gap either way. This measures something the size metric was never designed to catch: how tight/coherent each playlist actually is on the underlying features, averaged across all playlists in each pipeline. Lower = more internally consistent.


In [ ]:
def mean_within_cluster_std(cluster_col):
    # IMPORTANT: use scaled_all, not df[features] directly — tempo (60-200 BPM) on the
    # raw scale would otherwise dominate the average, the same bug caught earlier in the
    # duplicate-title check (see notebook 04 / report Section 17)
    temp = scaled_all.copy()
    temp[cluster_col] = df[cluster_col].values
    stds = temp.groupby(cluster_col)[features].std().mean(axis=1)
    return stds.mean()

baseline_coherence = mean_within_cluster_std('baseline_cluster')
hybrid_coherence = mean_within_cluster_std('hybrid_cluster')

print(f"Single-stage — mean within-playlist std: {baseline_coherence:.4f}")
print(f"Hybrid       — mean within-playlist std: {hybrid_coherence:.4f}")
print(f"\nHybrid is {'tighter' if hybrid_coherence < baseline_coherence else 'looser'} by "
      f"{abs(baseline_coherence - hybrid_coherence) / baseline_coherence * 100:.1f}%")

**Also worth checking directly — did the single-stage pipeline ever accidentally straddle two very different Agglomerative superclusters within one playlist** (something the hybrid pipeline structurally cannot do, since it splits within superclusters only)? If baseline playlists frequently mix songs from clusters as different as "aggressive rock" (supercluster 1) and "classical" (supercluster 3), that's a concrete, specific coherence failure the hybrid approach prevents by construction.


In [ ]:
# for each single-stage playlist, how many distinct Agglomerative superclusters does it span?
spread = df.groupby('baseline_cluster')['agg_supercluster'].nunique()
print(spread.value_counts().sort_index())
print(f"\n{(spread > 1).sum()} out of {len(spread)} single-stage playlists span more than one supercluster")

## 9. Save Outputs


In [ ]:
df.to_csv("../outputs/songs_hybrid_clusters.csv", index=False)
comparison.to_csv("../outputs/06_hybrid_comparison.csv", index=False)
print("Saved.")

## 10. Verdict

*(Fill in once run — did starting from Agglomerative's 8 coherent superclusters improve on the single-stage pipeline's in-range % or size distribution, or land in roughly the same place? Either answer is a legitimate, reportable finding — a genuine improvement supports a two-stage recommendation; no meaningful difference confirms the single-stage pipeline was already capturing the important structure on its own.)*
